# BY from precomputed Jina-v4 cosine scores

This notebook downloads frozen cosine scores for 1,000 calibration queries and the complete official labelled test split of HotpotQA, MMQA, TAT-QA, and WebQA. It runs candidate-wise BY and the alpha sweep without downloading images, loading an embedding model, or using a GPU.

The rowwise files retain the original retrieval-log schema and pipeline fingerprints. The false-score banks use calibration rows only; all reported selections use the disjoint official test rows.


In [ ]:
import os, shutil, subprocess, sys, urllib.request
from pathlib import Path

REPO_URL = "https://github.com/NguyenKhanh2603/Uncertainty-Aware-Iterative-RAG.git"
CODE_COMMIT = "bc4cc9f"
PROJECT_DIR = Path("/content/uncertainty-aware-rag")
WORK_DIR = Path("/content/precomputed_cosine_by")
WORK_DIR.mkdir(parents=True, exist_ok=True)

uv = shutil.which("uv")
if uv is None:
    subprocess.check_call(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", "/tmp/install-uv.sh"])
    subprocess.check_call(["sh", "/tmp/install-uv.sh"])
    uv = str(Path.home() / ".local/bin/uv")
subprocess.check_call([uv, "pip", "install", "--system", "numpy>=1.26", "tqdm>=4.66"])
if not (PROJECT_DIR / ".git").is_dir():
    subprocess.check_call(["git", "clone", "--filter=blob:none", REPO_URL, str(PROJECT_DIR)])
subprocess.check_call(["git", "-C", str(PROJECT_DIR), "fetch", "origin", CODE_COMMIT])
subprocess.check_call(["git", "-C", str(PROJECT_DIR), "checkout", "--detach", CODE_COMMIT])
os.environ["PYTHONPATH"] = str(PROJECT_DIR / "src") + os.pathsep + str(PROJECT_DIR)
print("Ready. GPU is not required.")


## Download the precomputed query scores

Each gzip JSONL contains both roles under one fingerprint: 1,000 calibration queries followed by the complete official test split. Scores were produced with pinned `jinaai/jina-embeddings-v4`, Top-L up to 30, and dataset-provided candidate pools.


In [ ]:
DATASETS = ["hotpotqa", "mmqa", "tatqa", "webqa"]
BRANCH = "results/qwen2vl-jina4-four-datasets-2026-09-14"
BASE = (
    "https://raw.githubusercontent.com/NguyenKhanh2603/"
    "Uncertainty-Aware-Iterative-RAG/refs/heads/" + BRANCH +
    "/research/internal_state_rag/results/"
    "precomputed_cosine_cal1000_official_test_2026_09_30/portable_bundle"
)
ROW_DIR = WORK_DIR / "rowwise"
ROW_DIR.mkdir(exist_ok=True)
log_paths = []
for dataset in DATASETS:
    name = f"{dataset}_top30_retrieval.jsonl.gz"
    for relative in (f"rowwise/{name}", f"rowwise/{name}.manifest.json"):
        target = WORK_DIR / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        print("Downloading", relative)
        urllib.request.urlretrieve(f"{BASE}/{relative}", target)
    log_paths.append(ROW_DIR / name)
bank_path = WORK_DIR / "combined_reference_banks.json.gz"
urllib.request.urlretrieve(f"{BASE}/combined_reference_banks.json.gz", bank_path)
print("Downloaded", [p.name for p in log_paths])


## Run candidate-wise BY on official test queries


In [ ]:
ALPHA = 0.10
MAX_CONTEXT = 10
selection_dir = WORK_DIR / "selection"
command = [
    sys.executable, str(PROJECT_DIR / "scripts/run_conformal_backfill_selection.py"),
    "--input", *map(str, log_paths),
    "--bank", str(bank_path),
    "--output-dir", str(selection_dir),
    "--split-role", "test",
    "--alpha", str(ALPHA),
    "--max-context", str(MAX_CONTEXT),
    "--allow-underpowered-banks",
    "--compare-pooled",
]
subprocess.check_call(command, cwd=PROJECT_DIR, env=os.environ.copy())
import json
summary = json.loads((selection_dir / "selection_summary.json").read_text())
for strategy, values in summary["strategies"].items():
    print(strategy, {key: values[key] for key in (
        "queries", "average_selected_chunks", "empty_context_rate",
        "micro_evidence_precision", "conditional_reserve_support_recall"
    )})


## Replay the alpha sweep without encoding


In [ ]:
sweep_dir = selection_dir / "alpha_sweep"
subprocess.check_call([
    sys.executable, str(PROJECT_DIR / "scripts/sweep_conformal_selection_alpha.py"),
    "--input", str(selection_dir / "selection_decisions.jsonl.gz"),
    "--output-dir", str(sweep_dir),
    "--alphas", "0.01,0.025,0.05,0.10,0.20,0.30,0.50,0.75,0.90",
    "--max-context", str(MAX_CONTEXT),
], cwd=PROJECT_DIR, env=os.environ.copy())
print((sweep_dir / "alpha_sweep_summary.csv").read_text()[:5000])


## Download the BY outputs


In [ ]:
archive = shutil.make_archive("/content/precomputed_cosine_BY_results", "zip", selection_dir)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass
